In [1]:
import pennylane as qml
from pennylane import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

# Load and preprocess data
df = pd.read_csv('Algerian_forest_fires_cleaned.csv')
X = df.drop(['day', 'year', 'FWI', 'Classes'], axis=1).values
y = df['Classes'].values.astype(float)  # Convert to float

# Convert labels to -1.0 and 1.0 (must be floats for differentiation)
y = np.where(y == 0, -1.0, 1.0)

# Scale features to [0, π] range for quantum encoding
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X).astype(float)  # Ensure float type

# Reduce feature dimension to 4 for manageable circuit size
X_reduced = X_scaled[:, :4]  # Take first 4 features

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_reduced, y, test_size=0.2, random_state=42, stratify=y)

# Quantum circuit parameters
n_qubits = 4  # Matches our reduced feature dimension
n_layers = 3   # Number of variational layers

# Quantum device (using default simulator)
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="autograd")
def quantum_circuit(inputs, weights):
    # Feature embedding
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)
    
    # Variational layers
    for layer in range(n_layers):
        for i in range(n_qubits):
            qml.RY(weights[layer, i], wires=i)
            qml.RZ(weights[layer, i + n_qubits], wires=i)
        for i in range(n_qubits):
            qml.CNOT(wires=[i, (i+1)%n_qubits])
    
    # Measurement
    return qml.expval(qml.PauliZ(0))

# Variational classifier
def variational_classifier(weights, x):
    return quantum_circuit(x, weights)

# Cost function
def cost(weights, X, Y):
    predictions = np.array([variational_classifier(weights, x) for x in X])
    return np.mean((Y - predictions) ** 2)

# Initialize weights (must be float and require grad)
weights = np.random.uniform(0, 2*np.pi, (n_layers, 2*n_qubits), requires_grad=True)

# Training
opt = qml.GradientDescentOptimizer(stepsize=0.1)

for epoch in range(50):
    # Ensure we're passing float arrays
    weights, current_cost = opt.step_and_cost(
        lambda w: cost(w, X_train.astype(float), y_train.astype(float)), 
        weights
    )
    if epoch % 5 == 0:
        print(f"Epoch {epoch}: Cost = {current_cost:.4f}")

# Evaluation
def predict(weights, X):
    return np.sign([variational_classifier(weights, x) for x in X])

train_pred = predict(weights, X_train)
test_pred = predict(weights, X_test)

# Convert predictions back to 0/1 for accuracy calculation
train_pred = np.where(train_pred == -1, 0, 1)
test_pred = np.where(test_pred == -1, 0, 1)
y_train = np.where(y_train == -1, 0, 1)
y_test = np.where(y_test == -1, 0, 1)

train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)

print(f"\nTraining Accuracy: {train_acc:.2f}")
print(f"Test Accuracy: {test_acc:.2f}")

# Visualize the circuit
print("\nQuantum Circuit:")
print(qml.draw(quantum_circuit)(X_train[0], weights))

Epoch 0: Cost = 1.1487
Epoch 5: Cost = 1.0896
Epoch 10: Cost = 1.0555
Epoch 15: Cost = 1.0336
Epoch 20: Cost = 1.0174
Epoch 25: Cost = 1.0041
Epoch 30: Cost = 0.9922
Epoch 35: Cost = 0.9810
Epoch 40: Cost = 0.9698
Epoch 45: Cost = 0.9586

Training Accuracy: 0.57
Test Accuracy: 0.57

Quantum Circuit:
0: ──RY(2.09)──RY(3.64)──RZ(2.83)─╭●───────╭X──RY(5.42)──RZ(5.13)─╭●───────╭X──RY(3.83) ···
1: ──RY(1.88)──RY(0.26)──RZ(1.70)─╰X─╭●────│───RY(6.27)──RZ(3.10)─╰X─╭●────│───RY(0.59) ···
2: ──RY(0.96)──RY(1.80)──RZ(1.82)────╰X─╭●─│───RY(5.26)──RZ(2.04)────╰X─╭●─│───RY(4.80) ···
3: ──RY(1.50)──RY(1.41)──RZ(4.01)───────╰X─╰●──RY(1.56)──RZ(0.14)───────╰X─╰●──RY(3.31) ···

0: ··· ──RZ(6.03)─╭●───────╭X─┤  <Z>
1: ··· ──RZ(1.98)─╰X─╭●────│──┤     
2: ··· ──RZ(4.52)────╰X─╭●─│──┤     
3: ··· ──RZ(4.28)───────╰X─╰●─┤     
